In [41]:
import scanpy as sc
import scarches as sca
import numpy as np

In [ ]:
# Path to the folder containing the files
data_path = "/shares/vasciaveo_lab/data/nepc_organoid_project/new_data/MJ005/filtered_feature_bc_matrix"

# Load the 10X Genomics formatted data
adata = sc.read_10x_mtx(
    data_path,              # Directory with the matrix.mtx, barcodes.tsv, genes.tsv files
    var_names='gene_symbols',  # Use gene symbols as variable names
    cache=True                 # Cache the result for faster loading next time
)

adata.layers['counts'] = adata.X.copy()

adata_marker_path = '/home/lgolinelli/git/GRN-VAE/expimap_outputs/adata_markers.h5ad'
adata_marker = sc.read_h5ad(adata_marker_path)

if (adata_marker.obs_names == adata.obs_names).all():
    clusters_path = '/home/lgolinelli/git/GRN-VAE/outputs/acdc_pax_3_clusters.npy'
    adata.obs['cancer_subtype'] = adata_marker.obs['cancer_subtype'].values.copy()
else:
    raise ValueError("Observation names do not match between adata and adata_marker.")

del adata_marker

In [43]:
import pandas as pd

In [48]:
annotation_path = '/home/lgolinelli/git/scarches-1/notebooks/GRN-VAE/binary_matrices/MJ005_binary_matrix_1%.csv'
df =pd.read_csv(annotation_path)
df

,regulator,0610009B22Rik,0610009O20Rik,0610010F05Rik,0610010K14Rik,0610012G03Rik,0610025J13Rik,0610030E20Rik,0610033M10Rik,0610037L13Rik,...,mt-Co2,mt-Co3,mt-Cytb,mt-Nd1,mt-Nd2,mt-Nd3,mt-Nd4,mt-Nd4l,mt-Nd5,mt-Nd6
0,0610010K14Rik,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1700020N01Rik,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2010315B03Rik,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2610008E11Rik,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2610021A01Rik,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2527,Zscan26,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2528,Zscan29,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2529,Zxdb,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2530,Zxdc,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [50]:
np.where(df.regulator == 'Nsd2')

(array([1299]),)

In [60]:
Nsd2_targets = df.columns[np.where(df.iloc[np.where(df.regulator == 'Nsd2')] == 1)[1]]

In [64]:
Nsd2_targets_in_adata = [name for name in adata.var_names if name in Nsd2_targets]

In [65]:
len(Nsd2_targets_in_adata)

31

In [78]:
annotation_path = '/home/lgolinelli/git/scarches-1/notebooks/GRN-VAE/binary_matrices/tsv_outputs/regulator_gene_1percent.tsv'
sca.utils.add_annotations(adata, annotation_path, min_genes=1, clean=False, genes_use_upper=False)
adata

AnnData object with n_obs × n_vars = 4544 × 31053
    obs: 'cancer_subtype'
    var: 'gene_ids'
    uns: 'terms'
    varm: 'I'
    layers: 'counts'

In [76]:
df.shape

(2532, 16343)

In [40]:
'Nsd2' in list(adata.uns['terms'])

False

In [ ]:
adata._inplace_subset_var(adata.varm['I'].sum(1)>0)


In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers['X_normalized'] = adata.X.copy()

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    subset=True)



In [ ]:
# Filter out any annotations (terms) with less than 12 genes.
select_terms = adata.varm['I'].sum(0)>1
adata.uns['terms'] = np.array(adata.uns['terms'])[select_terms].tolist()
adata.varm['I'] = adata.varm['I'][:, select_terms]

# Filter out genes that are not annotated to any term after HVG selection
adata._inplace_subset_var(adata.varm['I'].sum(1)>0)


adata.X = adata.layers["counts"].copy()

In [ ]:
import os
pic_path = '/home/lgolinelli/git/scarches-1/notebooks/GRN-VAE/pics/'
os.makedirs(pic_path, exist_ok=True)
for layer in ['X_normalized', 'counts']:
    adata.X = adata.layers[layer].copy()
    sc.pp.pca(adata)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)
    sc.pl.umap(adata, color='cancer_subtype', title=layer, save=f'{pic_path}_{layer}_cancer_subtype.png')

In [ ]:
'Nsd2' in adata.var_names

False

In [14]:
'Nsd2' in adata.uns['terms']

False

In [ ]:
adata.varm['I'].shape
'Nsd2' in adata.uns['terms']

(1994, 2212)

In [18]:
adata.var_names

Index(['4732440D04Rik', 'Trpa1', 'Terf1', '4930444P10Rik', 'Ly96', 'Pkhd1',
       'Efhc1', 'Bag2', 'Gm38336', '1110002O04Rik',
       ...
       'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb', 'Vamp7',
       'AC168977.1', 'AC149090.1', 'CAAA01147332.1'],
      dtype='object', length=1994)

In [24]:
adata.var['gene_idx'] = np.arange(adata.shape[1])

In [ ]:
adata.var['gene_idx'] = np.arange(adata.shape[1])
adata.var['counts_in_term'] = 0
adata.var['gene_as_regulator'] = False
for gene_name in adata.var_names:
    gene_index = adata[:, gene_name].var['gene_idx']
    gene_count_in_term = adata.varm['I'][gene_index].sum()
    adata.var.loc[gene_name, 'counts_in_term'] = gene_count_in_term
    adata.var.loc[gene_name, 'gene_as_regulator'] = gene_name in adata.uns['terms']

if 'Nsd2' in adata.var_names:
    adata.uns['Nsd2_as_regulator'] = True
else:
    adata.uns['Nsd2_as_regulator'] = False
